# Chapter 05: Runners: Hosted, Larger, Self-Hosted, ARC (notebook edition)

## Learning Objectives

- Compare measured runners to the docs
- Price a mixed-OS matrix
- Tell skipped from stuck

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Load the measured runners

The fixture holds the `REPORT` line each runner printed in run 37312789542. We compare CPU and memory with the docs' public-repo table.

In [ ]:
import json
data = json.loads((ROOT/"fixtures"/"run_ch05_runners.json").read_text())
by = {j["label"]: j for j in data["jobs"]}
for label, (cpu, gb) in data["docs_public_specs"].items():
    print(label, by[label]["cpu"], round(by[label]["mem_mb"]/1024, 1), "GB | docs:", cpu, gb)
    assert by[label]["cpu"] == cpu

## 2. Price the matrix

Each job rounds up to one minute and is charged at its OS rate. This repo is public so the real bill is $0; this is the private-repo equivalent. You should see 84 mills.

In [ ]:
from datetime import datetime
from intro_gha.cost import run_cost_mills, job_cost_mills
FMT = "%Y-%m-%dT%H:%M:%SZ"
OS = {"Linux": "linux", "Windows": "windows", "macOS": "macos"}
sec = lambda j: int((datetime.strptime(j["completed_at"], FMT) - datetime.strptime(j["started_at"], FMT)).total_seconds())
jobs = [(sec(j), OS[j["os"]]) for j in data["jobs"]]
print(jobs, run_cost_mills(jobs))
assert run_cost_mills(jobs) == 84
print("macOS share %:", round(job_cost_mills(5, "macos") / 84 * 100))

## 3. The guarded job and the stuck job

The guarded job skipped in 0 seconds. The job asking for `tally-gpu` stayed queued with no runner.

In [ ]:
g = data["guarded_job"]
print(g["conclusion"], g["started_at"] == g["completed_at"])
stuck = json.loads((ROOT/"fixtures"/"run_ch05_no_runner.json").read_text())
print(stuck["job_status"], stuck["job_runner_name"])
assert g["conclusion"] == "skipped" and stuck["job_status"] == "queued" and stuck["job_runner_name"] is None

## 4. Exercise 5 in code

3 Linux jobs of 70 s plus 1 Windows job of 20 s.

In [ ]:
mix = [(70, "linux")] * 3 + [(20, "windows")]
print(run_cost_mills(mix))
assert run_cost_mills(mix) == 46

## Takeaways & Next Steps

- Public repos get bigger hosted machines than private ones.
- macOS dominates mixed-OS bills.
- A guarded job skips; an unguarded one with no runner waits silently.
- Next: Chapter 06, events and triggers in depth (Phase 2).

## Chapter Link

Read: `learning_modules/chapter_05_runners.md`